<a href="https://colab.research.google.com/github/number1coder01/Machine-Learning/blob/main/Lecture28ColumnTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd

In [4]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [5]:
df = pd.read_csv('covid_toy.csv')

In [6]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [7]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [8]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],
                                                test_size=0.2)

In [9]:
X_train

,age,gender,fever,cough,city
68,54,Female,104.0,Strong,Kolkata
65,69,Female,102.0,Mild,Bangalore
79,48,Female,103.0,Mild,Kolkata
94,79,Male,NaN,Strong,Kolkata
52,47,Female,100.0,Strong,Bangalore
...,...,...,...,...,...
41,82,Male,NaN,Mild,Kolkata
23,80,Female,98.0,Mild,Delhi
15,70,Male,103.0,Strong,Kolkata
16,69,Female,103.0,Mild,Kolkata



## 1. Without column transformer

In [10]:
# adding simple imputer to fever col (missing values ke liye)
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']]) # data ke mean se replaced isme

# also the test data
X_test_fever = si.fit_transform(X_test[['fever']])

X_train_fever.shape

(80, 1)

In [12]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [15]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also the test data
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape

(80, 4)

In [16]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

In [17]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

## With column transformer

In [18]:
from sklearn.compose import ColumnTransformer

In [20]:
# takes in 2 inputs -> kis kis par transformation lagani as a list and next is remainder i.e. baaki bache huye columns ka kya karna jinpar transfomn nahi lagani
# transformers sent as tuple -> iss tuple mei 3 cheeze sent -> 1. transformer ka name 2. konsa transformer? 3. list of columns jinpar apply karna hai
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')

In [21]:
transformer.fit_transform(X_train).shape

(80, 7)

In [22]:
transformer.transform(X_test).shape

(20, 7)